In [1]:
import sounddevice as sd
from scipy.io.wavfile import write

def record_audio(filename="audio.wav", duration=5):
    sample_rate = 16000

    print("Listening...")

    audio = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="int16"
    )

    sd.wait()

    write(filename, sample_rate, audio)

    print("Recording complete.")

In [2]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

def transcribe_audio(filename="audio.wav"):
    segments, info = whisper_model.transcribe(filename)

    text = " ".join(
        segment.text for segment in segments
    )

    return text.strip()

c:\Users\anves\OneDrive\Desktop\Voice Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datetime import datetime

def get_current_time():
    return datetime.now().strftime("%I:%M %p")


def calculate(expression):
    try:
        return str(eval(expression))
    except Exception:
        return "Unable to calculate the expression."

In [4]:
import ollama
import json

tools_description = """
Available tools:

1. get_current_time
Use this when the user asks for the current time.

2. calculate
Use this for mathematical calculations.

If a tool is required, respond ONLY with JSON:

{
    "tool": "tool_name",
    "argument": "tool_argument"
}

If no tool is required, respond normally.
"""

In [5]:
def ask_llm(user_text):
    prompt = f"""
You are a helpful Voice AI Agent.

{tools_description}

User request:
{user_text}
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [6]:
def process_response(response):
    try:
        tool_call = json.loads(response)

        tool_name = tool_call.get("tool")
        argument = tool_call.get("argument")

        if tool_name == "get_current_time":
            result = get_current_time()

        elif tool_name == "calculate":
            result = calculate(argument)

        else:
            return response

        return f"The result is {result}"

    except json.JSONDecodeError:
        return response

In [7]:
import subprocess

PIPER_MODEL = "en_US-lessac-medium.onnx"

def speak(text):
    command = [
        "piper",
        "--model",
        PIPER_MODEL,
        "--output_file",
        "response.wav"
    ]

    process = subprocess.Popen(
        command,
        stdin=subprocess.PIPE,
        text=True
    )

    process.communicate(text)

    subprocess.run([
        "python",
        "-c",
        """
import sounddevice as sd
from scipy.io.wavfile import read

rate, audio = read('response.wav')
sd.play(audio, rate)
sd.wait()
"""
    ])

In [12]:
def run_voice_agent():
    while True:
        record_audio()

        user_text = transcribe_audio()

        print("You:", user_text)

        if user_text.lower() in ["exit", "quit", "stop"]:
            print("Voice Agent stopped.")
            break

        llm_response = ask_llm(user_text)

        final_response = process_response(llm_response)

        print("Agent:", final_response)

        speak(final_response)


if __name__ == "__main__":
    run_voice_agent()

Listening...
Recording complete.
You: What is AI?
Agent: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions.
Listening...


KeyboardInterrupt: 